# పాఠం 18 (తదుపరి): ఒక *మానవుడు* చర్యను অনুমతించారని నిరూపించే రసీదులు

పాఠం **ఏజెంట్** ఏమి చేశాడో మరియు **గేట్** ఏమి నిర్ణయించిందో చూపిస్తుంది. ఈ నోట్‌బుక్ లోపలి లేకపోయే పార్ట్‌ని జోడిస్తుంది: ఒక **పేరున్న మానవుడు** **సరిగ్గా** ఆ చర్యకు ఆమోదం ఇచ్చారని ప్రూఫ్ — పూర్తి కెనానికల్ చర్యపై వేరే, మానవుడి చేతి సంతకం, ఆన్‌లైన్ వెలుపల ధృవీకరణ.

ఇక్కడ ఉన్న ఇద్దరు ఆర్టిఫాక్ట్లు రెండూ పాఠంలోని రసీదుల **అదే సంపుటి ఆకారాన్ని ఉపయోగిస్తాయి**: ఒక ఫ్లాట్ పేలెడ్, `type` ఫీల్డ్‌తో, ఏడీ25519 చేత సంతకం చేయబడింది, కెనానికల్ పేలెడ్ బైట్స్ పై SHA-256 తో, సంస్కృతిభావమైన `signature` ఆబ్జెక్ట్ జోడించబడింది (మరియు సంతకం చేసిన బైట్స్ నుండి తప్పించబడింది). ఆమోద రసీదు కొత్త `type` (`human.approval.v1`)గా చర్య టైప్ పక్కన ఉంటుంది, కాబట్టి ఒకటి `verify_chain` రెండు ఆర్టిఫాక్ట్ రకాలని అదే కోడ్ పాథ్ తో main notebook లో మీరు తయారుచేసిన దానితో కవర్ చేస్తుంది. ఇది పాఠం అనుసరించిన ఇంటర్నెట్-డ్రాఫ్ట్ లోని సహి మార్గం యొక్క ఆకారంగా కూడా ఉంటుంది (draft-farley-acta-signed-receipts).

main notebook లో ఉన్న డెమో వెరిఫయర్ పై ఒకసారైన పద్ధతిగా: ఇక్కడ వెరిఫయర్ `signature.key_id` ని రసీదులో ఉన్న ప్రజా కీ మీద ఆధారపడి ఉండటం కాకుండా **పిన్న్డ్ కీ రిజిస్ట్రీ** మీద కదలిస్తుంది. అది ప్రొడక్షన్ దృక్కోణం పాఠం యొక్క తన సిన్ చెక్‌లిస్ట్ సిఫారసు చేసే దాని విధానం ("ధృవీకరణ ప్రజా కీ ప్రచురించండి"), మరియు ఇది మంచితనంతో కలిపే తప్పించుకోకుండా నకలీకరణను నివారిస్తుంది.

ఈ నోట్‌బుక్ బోధించే నియమం: **ఒక సంతకం చేసిన ఆమోదం స్వయంగా అధికారం కాదు.** అధికారం కేవలం ఆ ఆమోద రసీదు మరియు చర్య రసీదు అమలు సమయానికి అదే కెనానికల్ చర్యకు ఇది కూడా ఏర్పాటు చేస్తేనే ఉంటుంది, ఒక విధాన సంస్కరణ, కీ మరియు గడువు ఇప్పటికీ చెలామణీలో ఉంటే, మరియు ఆమోదం ఇప్పటికే వినియోగించబడకపోతేనే. ప్రతి విఫలం **విశిష్ట కారణంతో** తిరస్కరించబడుతుంది, కనుక మీరు *అధికారం పాతదై ఉందని* మరియు *చర్య మారిపోయిందని* వేరుచేసుకోవచ్చు.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## ఖచ్చితమైన చర్య  

ఆమోదం యొక్క యూనిట్ **కానానికల్ చర్య వస్తువు** — "రిఫండ్ ఆమోదించు" వంటి అస్పష్ట లేబుల్ కాదు, కానీ ఖచ్చితమైన, పూర్తిగా నిర్దేశించిన చర్య. మొత్తం వస్తువుపై సంతకం చేయడం (మరియు దాని నుండి డైజెస్ట్ తయారుచేయడం) మనకు తరువాత మనిషి ఈ ఒక్కది ఆమోదించాడన proof సాదించడానికి సహాయం చేస్తుంది, మరేదీ కాదు.  


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ఒక లేఖనం, రెండు అధికారులు

ప్రతి రసీదు పాఠం యొక్క లేఖనం: ఒక ఫ్లాట్ పేలాడ్ `type` ఫీల్డ్‌తో పాటు, ఒక `signature` ఆబ్జెక్ట్ (`alg`, `sig`, `key_id`) తో ఉంటుంది, ఇది సంతకం చేసిన బైట్లలో భాగం **కాదు**. `verify_envelope` రెండు రసీదు రకాల కోసం పంచుకున్న నిర్మాణ + సంతకం చెక్; ఇది `signature.key_id` ను జవాబుదారీ అధికారుల మధ్య వేరుచేసే **పిన్ చేసిన కీ రిజిస్ట్రీ** ను ఎలా పరిష్కరిస్తుందో అందిస్తుంది:

- **సమర్థన రసీదు** (`human.approval.v1`) — పేరు చెప్పబడిన సమర్థకుడు, పూర్తి కానానికల్ చర్య **మరియు దాని డైజెస్ట్**, `policy_version`, జారీ + కాలపరిమితి టైమ్స్టాంప్లు. చైన్ స్థాయిలో ఒకసారి వినియోగంకు ట్రాక్ చేయబడుతుంది.
- **చర్య రసీదు** (`agent.action.v1`) — ఏజెంట్ గుర్తింపు, `run_id`, అదే కానానికల్ చర్య **డైజెస్ట్**, అమలులో ఫలితం + టైమ్స్టాంప్ మరియు `parent_approval_ref`: సమర్థన యొక్క `receipt_hash`, పాఠం చైన్ లో `previous_receipt_hash` అనే కొత్త విధానం.

పంచుకున్న `action_digest` ఫీల్డ్ అనేది బైండింగ్ మీద ఆధారపడేది. `key_id` సంతకం ఆబ్జెక్టులో పరామర్శ సూచనగా మాత్రమే ఉంటుంది: దాన్ని వేరే పిన్ చేసిన కీకి మార్చడం సంతకం చెక్‌ను విఫలమవ్విస్తుంది, కాబట్టి ఇది ఎటువంటి ప్రయోజనం ఇవ్వదు.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: బైండింగ్ నిజంగా నిర్ణయించబడే స్థలం

`verify_chain` రెండు సంతకం చెక్‌ల పై సౌలభ్యమైన ర్యాపర్ కాదు. ఇది ఒకే చోటే ఉంది అందివ్వబడిన షేర్ చేయబడిన ప్రామాణిక `action_digest`, ఆమోదం యొక్క పాలసీ/కీ/కాల పరిమితి **తాజావడం**, మరియు ఆమోదం యొక్క **ఒకసారి వినియోగం** ప్రమాదంలో ఉన్న చర్యకు *ఇప్పుడే* అమలు చేయబడే చర్యకు వ్యతిరేకంగా పరీక్షించబడతాయి.

ప్రతి విఫలం ఒక **విభిన్న కారణంతో** తిరస్కరిస్తుంది, కాబట్టి తిరస్కరణ చదివే వ్యక్తి అధికారికత సడలిపోయిందని (పాలసీ మారిపోయింది, కీ మార్పిడి చేయబడింది, ఆమోదం గడువు ముగిసింది, ఆమోదం వినియోగించబడింది) లేదా అమలు చేయబడిన చర్య ఇప్పటికీ చెల్లిపోయే ఆమోదం కింద మార్చబడిందని (డైజెస్ట్ ప్రత్యామ్నాయం) ఏంటో తెలుపుతుంది.


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## బైండింగ్ ఏమి పట్టుకుంటుంది

క్రింది ప్రతి కేసు ఒక **విభిన్న కారణంతో** **మూసివేయబడుతుంది**. మొదటి బ్లాక్ క్లాసిక్ సెట్ను సూచిస్తుంది (తప్పుడు చర్య, గందరగోళమైన డిప్యూటీ, మళ్లింపు, ఎంత అధికారంపై ఉన్నా جعل سازی, ఆపర malformed ఇన్‌పుట్). రెండవ బ్లాక్ ఆ జంటను సూచిస్తుంది దీని వల్ల ప్రాపర్టీ నిర్దిష్టంగా ఉండటం కాదు, కానీ నిజంగా అమలయ్యేలా చేస్తుంది:

- **పాత అధికారం** — సంతకం ఇంకా有効ంగా ఉంటుంది, కానీ విధాన సంచిక మారిపోయింది, అభ్యర్థించే వ్యక్తి కీ pinned రిజిస్ట్రీలోనుంచి మార్చబడింది, లేదా ఆమోదం అమలు కంటే ముందు గడువు తీరింది;
- **డైజెస్ట్ స్థానాంతరం** — సరైన సంతకం చేసిన చర్య రసీదు; దీని `parent_approval_ref` ఒక *నిజమైన* ఆమోదాలను సూచిస్తుంది, గాని ఆ ఆమోదం యొక్క కనూనికల్ చర్య డైజెస్ట్ అసలు అమలవుతున్న చర్యకి సరిపోలదు.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## ఇది నిదర్శనం చేసే విషయం — మరియు ఇది చేయని విషయం

**నిదర్శనం చేస్తుంది:** పేరున్న ఒక మనిషి ఆమోదించింది *ఈ ఖచ్చితమైన కెనానికల్ చర్యను* (పూర్తి చర్య + డైజెస్ట్, పిన్ చేసిన రిజిస్ట్రి నుండి తేలిన కీతో సంతకం), మరియు ఏజెంట్ *కచ్చితంగా ఆ ఆమోదించబడిన చర్యను* (అదే డైజెస్ట్, ఆమోదానికి `receipt_hash` ద్వారా బంధించబడిన రసీదు, పాఠం తన సొంత చైన్ కన్వెన్షన్) అమలు చేసింది — ఆమోద నిబంధన వెర్షన్, కీ, మరియు గడువు ఇప్పటికీ చలాకీ ఉన్నప్పుడు, ఒక్కసారి. ఏదైనా పక్షం మారితే, చైన్ మూసివేస్తుంది, మరియు తిరస్కరణ కారణం మీకు **ఏ** గుణం విరిగింది అనేది చెబుతుంది: పాత అధికారతా వర్సెస్ మారిన చర్య.

**నిదర్శనం చేయదు:** ఆమోద UI మనిషికి వారు సంతకం చేస్తున్నదని భావించినదాన్ని చూపించింది (WYSIWYS అనేది తనదైన సమస్య), కీ మార్చుకునే ముందు బలవంతంగా తీసుకోబడలేదు లేదా కొల్లబట్టలేదు, లేదా దిగువ ప్రభావాలు చర్యకు సరిపోతున్నాయి. సంతకం చేయబడింది అంటే అనుమతించబడింది కాదు: పాత పాలసీపై చెల్లుబాటు అయ్యే సంతకం, మార్చిన కీ, గడువు అయ్యే సమయంలో, లేదా వేరే డైజెస్ట్ వద్ద ఎలాంటి విలువ లేదు.

ఈ రెండు రసీదు రకాలు పాఠం మాండ్‌ను మరియు ఒక `verify_chain` కోడ్ మార్గాన్ని ఉద్దేశపూర్వకంగా పంచుకుంటాయి: మీరు ప్రధాన నోట్ల పుస్తకంలో చర్య రసీదుల కోసం నిర్మించిన బంధం మనిషి ఆమోదాన్ని సరిచూడే అదే కోడ్. ఒక ధృవీకరణ ఒప్పందం, వేరు చేయబడిన పిన్ చేసిన అధికారతలు, కెనానికల్ చర్య డైజెస్ట్ తో కలుసుకుని మరేదీ కాదు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
